# NeoMint Planner -- Adapter Merge, GGUF Conversion & Quantization

Converts the fine-tuned **NeoMint-Planner-1.7B** LoRA adapter into an optimized **GGUF (Q4_K_M)** file ready for local inference with Ollama or llama.cpp on Linux Mint.

### Workflow Steps:
1. **Merge**: Merge LoRA adapter weights into full-precision (FP16) base model weights.
2. **Convert**: Export Hugging Face FP16 model to GGUF format using `llama.cpp`.
3. **Quantize**: Quantize FP16 GGUF to `Q4_K_M` (~1.1 GB, balanced speed and precision).
4. **Verify**: Smoke test the quantized GGUF with `llama-cli`.
5. **Deploy**: Instructions to register with Ollama using `neomint.modelfile`.

## 1. Install Dependencies

In [ ]:
!pip install -q transformers>=4.44.0 peft>=0.12.0 accelerate torch sentencepiece gguf protobuf
!apt-get update -qq && apt-get install -y -qq cmake build-essential

## 2. Mount Google Drive or Locate Adapter

Locate the saved adapter directory `neomint-planner-1.7b-lora` from notebook 02.

In [ ]:
import os
import shutil
from pathlib import Path

ADAPTER_DIR = "neomint-planner-1.7b-lora"
BASE_MODEL = "Qwen/Qwen3-1.7B"
MERGED_DIR = "neomint-planner-1.7b-merged-f16"

if not os.path.exists(ADAPTER_DIR):
    drive_adapter = f"/content/drive/MyDrive/neomint/{ADAPTER_DIR}"
    if os.path.exists(drive_adapter):
        print(f"Copying adapter from Google Drive: {drive_adapter}")
        shutil.copytree(drive_adapter, ADAPTER_DIR, dirs_exist_ok=True)
    else:
        print("Adapter not found locally. Please upload adapter folder or mount Google Drive.")

## 3. Merge LoRA Adapter into Base Model (FP16)

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

print(f"Loading base model: {BASE_MODEL} in float16...")
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)

base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    torch_dtype=torch.float16,
    device_map="cpu",  # CPU avoids VRAM limits during full weight merging
    trust_remote_code=True,
)

print(f"Loading LoRA adapter from: {ADAPTER_DIR}...")
lora_model = PeftModel.from_pretrained(base_model, ADAPTER_DIR)

print("Merging LoRA weights into base model...")
merged_model = lora_model.merge_and_unload()

print(f"Saving merged FP16 model to: {MERGED_DIR}...")
merged_model.save_pretrained(MERGED_DIR, safe_serialization=True)
tokenizer.save_pretrained(MERGED_DIR)

print("Merge complete! Model saved as standard Hugging Face format.")

## 4. Setup llama.cpp & Build Conversion Binaries

In [ ]:
# Clone llama.cpp repository if not present
if not os.path.exists("llama.cpp"):
    !git clone --depth 1 https://github.com/ggerganov/llama.cpp.git

# Install python conversion requirements
!pip install -q -r llama.cpp/requirements.txt

# Build llama-quantize and llama-cli using cmake
!cmake -B llama.cpp/build -S llama.cpp -DGGML_NATIVE=OFF
!cmake --build llama.cpp/build --config Release -j --target llama-quantize llama-cli

## 5. Convert Merged Model to GGUF (FP16)

In [ ]:
GGUF_F16 = "neomint-planner-1.7b-f16.gguf"

print(f"Converting {MERGED_DIR} to {GGUF_F16}...")
# Run llama.cpp conversion script
!python llama.cpp/convert_hf_to_gguf.py {MERGED_DIR} --outfile {GGUF_F16} --outtype f16

if os.path.exists(GGUF_F16):
    size_mb = os.path.getsize(GGUF_F16) / (1024 * 1024)
    print(f"Converted GGUF (FP16) size: {size_mb:.1f} MB")
else:
    print("Conversion failed!")

## 6. Quantize GGUF to Q4_K_M (Ollama Target)

`Q4_K_M` provides the ideal compromise between inference speed, RAM usage (~1.1 GB), and JSON schema adherence.

In [ ]:
QUANT_TARGET = "neomint-planner-1.7b-Q4_K_M.gguf"
QUANT_BIN = "llama.cpp/build/bin/llama-quantize"
if not os.path.exists(QUANT_BIN):
    QUANT_BIN = "llama.cpp/llama-quantize"

print(f"Quantizing {GGUF_F16} -> {QUANT_TARGET} with Q4_K_M...")
!{QUANT_BIN} {GGUF_F16} {QUANT_TARGET} Q4_K_M

if os.path.exists(QUANT_TARGET):
    size_mb = os.path.getsize(QUANT_TARGET) / (1024 * 1024)
    print(f"\nSUCCESS! Quantized GGUF size: {size_mb:.1f} MB ({size_mb/1024:.2f} GB)")
else:
    print("Quantization failed!")

## 7. Smoke Test Inference with llama-cli

In [ ]:
CLI_BIN = "llama.cpp/build/bin/llama-cli"
if not os.path.exists(CLI_BIN):
    CLI_BIN = "llama.cpp/llama-cli"

SYSTEM_PROMPT = "You are NeoMint Planner, a local action-planning model for Linux Mint. Your only output must be a single valid JSON object following the NeoMint Action Plan schema. You may output kind: plan, clarification, or refusal."
TEST_PROMPT = f"<|im_start|>system\n{SYSTEM_PROMPT}<|im_end|>\n<|im_start|>user\nFind all PDFs in my Documents folder.<|im_end|>\n<|im_start|>assistant\n"

print("Running smoke test with llama-cli:")
!{CLI_BIN} -m {QUANT_TARGET} -p "{TEST_PROMPT}" -n 256 --temp 0.2 -c 2048

## 8. Download or Save to Google Drive

In [ ]:
if os.path.exists("/content/drive/MyDrive"):
    drive_dest = f"/content/drive/MyDrive/neomint/{QUANT_TARGET}"
    print(f"Copying {QUANT_TARGET} to Google Drive: {drive_dest}...")
    shutil.copy(QUANT_TARGET, drive_dest)
    print("Saved to Google Drive!")
else:
    from google.colab import files
    print("Triggering browser download for GGUF model...")
    files.download(QUANT_TARGET)

## 9. Next Steps: Local Deployment in NeoMint

Once downloaded to your local Linux Mint / development machine:

```bash
# 1. Place the GGUF file in the NeoMint repository root
cp neomint-planner-1.7b-Q4_K_M.gguf /path/to/NeoMint/

# 2. Create the Ollama model using neomint.modelfile
cd /path/to/NeoMint
ollama create neomint-planner -f neomint.modelfile

# 3. Test running directly with Ollama
ollama run neomint-planner "Find PDFs in my Documents folder"
```